[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/project-walkthrough/02_preprocess/02_preprocess.ipynb)

# 02. `preprocess-example` 동행 노트북

> 대상 프로젝트: [`example-projects/preprocess-example`](https://github.com/karzit/temp/tree/master/example-projects/preprocess-example) (A파트-2: 전처리/색인)
> · 앞 단계: [`01_crawl_storage`](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/project-walkthrough/01_crawl_storage/01_crawl_storage.ipynb)
> · 다른 선택지: [ALTERNATIVES.md](https://github.com/karzit/temp/blob/master/example-projects/preprocess-example/ALTERNATIVES.md)

## 이 장을 배우는 이유

앞 프로젝트가 원본을 DB에 쌓는 데서 끝났습니다. 그런데 그걸로는 아직 아무것도 못 합니다.

DB에 있는 건 "사람이 읽는 형태"일 뿐입니다. PDF와 HTML이 섞여 있고, 잡음이 끼어 있고,
문서 한 건이 통째로 들어 있습니다. **문서 전체가 검색 결과로 나오면 쓸모가 없습니다.**
200페이지짜리 규정집을 통째로 던져주면서 "여기 답 있어요"라고 하는 셈이니까요.

그래서 이 프로젝트가 세 가지를 합니다.

1. **형식 통일** — PDF든 HTML이든 뒤 단계는 "텍스트"만 받게
2. **정제** — 크롤링·PDF 추출에서 들어온 잡음 걷어내기
3. **[청킹](https://github.com/karzit/temp/blob/master/glossary.md#chunking)** — 검색할 수 있는 크기로 자르기

```
PostgreSQL 원본 -> 텍스트로 통일 -> 정제 -> 500자 청킹 -> 키워드 태깅 -> OpenSearch 색인
```

```
01 crawl-storage → [02 preprocess] → 03 document-input → 04 rag-regulation
                        여기
```

이번 장에서 배우는 것

- `RawDocument` dataclass가 딕셔너리보다 나은 이유
- 정제 정규식 4줄이 각각 무엇을 잡는지, **왜 청킹보다 먼저 해야 하는지**
- `io.BytesIO`로 PDF를 **파일로 저장하지 않고** 읽는 법
- 한국어에서 [형태소 분석](https://github.com/karzit/temp/blob/master/glossary.md#morphological-analysis)이 필요한 이유 (`휴가를` / `휴가는` 문제)
- [chunk_size](https://github.com/karzit/temp/blob/master/glossary.md#chunk-size-overlap)를 정하는 기준과 트레이드오프
- [임베딩](https://github.com/karzit/temp/blob/master/glossary.md#embedding)·[벡터 검색](https://github.com/karzit/temp/blob/master/glossary.md#vector-search)이 하는 일을 TF-IDF로 감 잡기

**소요 시간**: 40~50분. **PostgreSQL·OpenSearch·API 키 없이 끝까지 실행됩니다.**
DB는 SQLite로, 임베딩 검색은 TF-IDF로 대체합니다.

## 이 노트북을 읽는 법

- **셀을 위에서부터 순서대로 실행하세요**(`Shift + Enter`).
- **실행 결과는 저장되어 있지 않습니다.** 직접 실행해야 출력이 나타납니다.
- 첫 셀에서 패키지를 설치합니다. `kiwipiepy`(형태소 분석기)가 있어 **Colab 기준 1~2분** 걸립니다.
- 코드 셀 앞에는 **지금 무엇을 할 것인지**를 적어두었습니다. 셀 뒤에는 두 가지가 붙습니다 —
  `show()`로 프로젝트 소스를 펼친 뒤에는 **코드에서 짚을 곳**이, 실제로 돌려본 뒤에는
  **결과 읽는 법**이 나옵니다.
- [`01_crawl_storage`](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/project-walkthrough/01_crawl_storage/01_crawl_storage.ipynb)를 먼저 보면 좋지만, 안 봐도 따라갈 수 있습니다.
- 낯선 용어는 [glossary.md](https://github.com/karzit/temp/blob/master/glossary.md)에서 찾아보세요.
- **에러가 나거나 결과가 예상과 다르면** [troubleshooting.md](https://github.com/karzit/temp/blob/master/troubleshooting.md)를 먼저 보세요.
  설치 실패, 한글 깨짐, `NameError`, API 키, GPU 설정처럼 여러 노트북에서 반복되는 문제를 모아뒀습니다.

## 막혔을 때 — 이 노트북에서 자주 나오는 증상

| 증상 | 원인 | 해볼 것 |
|---|---|---|
| `AssertionError: 프로젝트 경로를 찾지 못했습니다` | **이 노트북은 실제 프로젝트 파일을 열어야 돌아갑니다.** Colab이라면 `git clone`이 실패했고, 로컬이라면 저장소 밖에서 노트북을 열었습니다 | Colab: 네트워크·프록시 확인 후 첫 셀 재실행. 로컬: 저장소를 통째로 받아 원래 폴더 구조 그대로 열기 |
| `show(...)`에서 `FileNotFoundError` | 파일명 오타이거나 프로젝트 구조가 바뀜 | `import os; print(os.listdir(SRC))`로 실제 파일 목록 확인 |
| import한 함수가 예전 동작을 한다 | 프로젝트 파일을 수정했지만 파이썬이 이미 불러둔 모듈을 재사용 | `import importlib; importlib.reload(모듈)` 또는 런타임 재시작 |
| `ModuleNotFoundError` — 프로젝트 모듈을 못 찾음 | `sys.path.insert(0, SRC)` 셀을 건너뜀 | 맨 위 준비 셀부터 순서대로 실행 |
| 첫 설치 셀이 오래 걸린다 | `kiwipiepy`(형태소 분석기)가 무거워 **Colab에서 1~2분** 걸립니다 | 정상입니다 |
| `ModuleNotFoundError: No module named 'fitz'` | PyMuPDF는 **설치 이름이 `pymupdf`, import 이름이 `fitz`** 입니다 | 설치 셀 재실행. `pip install fitz`는 전혀 다른 패키지이니 설치하지 마세요 |
| PostgreSQL·OpenSearch 관련 에러 | 이 노트북은 **인프라 없이** SQLite와 TF-IDF로 대체해 돌아갑니다 | 해당 코드는 `show()`로 읽기만 합니다 |
| PDF에서 뽑은 글자가 비어 있다 | 스캔본(이미지) PDF | 정상적인 실패입니다. 그런 PDF는 [OCR](https://github.com/karzit/temp/blob/master/glossary.md#ocr)이 필요합니다(03번 주제) |

여기 없는 문제(설치 실패, 한글 깨짐, API 키 설정 방법)는 [troubleshooting.md](https://github.com/karzit/temp/blob/master/troubleshooting.md)에 모아뒀습니다.

## 0. 환경 준비 — 프로젝트를 옆에 펼쳐두기

In [ ]:
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
print("Colab에서 실행 중:", IN_COLAB)

if IN_COLAB:
    # 이 노트북은 "예제 프로젝트를 옆에 두고 같이 읽는" 노트북입니다.
    # 그래서 설명만 하지 않고, 저장소를 통째로 내려받아 **실제 프로젝트 파일**을 열어봅니다.
    subprocess.run(["git", "clone", "-q", "https://github.com/karzit/temp.git", "/content/temp"], check=False)
    REPO_ROOT = "/content/temp"
    !pip install -q pymupdf kiwipiepy langchain-text-splitters langchain-core scikit-learn python-dotenv psycopg2-binary
else:
    # 로컬에서 열었다면 이 노트북 위치(notebooks/project-walkthrough/NN_xxx/)에서 3단계 위가 저장소 루트입니다.
    REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))

PROJECT = os.path.join(REPO_ROOT, "example-projects", "preprocess-example")
SRC = os.path.join(PROJECT, "src")
print("프로젝트 경로:", PROJECT)
assert os.path.isdir(SRC), "프로젝트 경로를 찾지 못했습니다. 저장소 루트에서 노트북을 열었는지 확인하세요."

아래 `show()`는 이 노트북 전체에서 쓰는 도우미입니다. **설명 대신 진짜 프로젝트 파일을 그대로 출력**해서, 노트북과 코드가 어긋나지 않게 합니다.

In [ ]:
import re


def show(filename, start=None, end=None, grep=None):
    """프로젝트 파일의 실제 소스를 줄 번호와 함께 출력한다.

    설명을 읽는 것과 실제 코드를 보는 것 사이의 간격을 없애기 위한 도우미입니다.
    이 노트북에서 "코드 읽기"라고 나오는 곳은 전부 진짜 프로젝트 파일을 그대로 보여줍니다.

        show("crawl.py")                  전체
        show("crawl.py", 30, 45)          30~45번째 줄
        show("crawl.py", grep="def ")     'def '가 들어간 줄만
    """
    path = os.path.join(SRC, filename) if not os.path.isabs(filename) else filename
    lines = open(path, encoding="utf-8").read().splitlines()

    if grep:
        picked = [(i, l) for i, l in enumerate(lines, 1) if re.search(grep, l)]
    else:
        s = (start or 1) - 1
        e = end or len(lines)
        picked = [(i, l) for i, l in enumerate(lines[s:e], s + 1)]

    for i, line in picked:
        print(f"{i:>4} | {line}")


def show_file(relpath, **kwargs):
    """프로젝트 루트 기준 경로로 파일을 보여준다 (README, docker-compose 등)."""
    show(os.path.join(PROJECT, relpath), **kwargs)


# 프로젝트 소스를 import할 수 있도록 경로를 등록해둡니다.
if SRC not in sys.path:
    sys.path.insert(0, SRC)

`config.py`가 `OPENAI_API_KEY`를 필수로 요구하기 때문에, import가 되도록 가짜 값을 넣어둡니다.
(임베딩을 실제로 호출하지는 않습니다 — 아래 8번에서 그 부분만 다른 방법으로 대체합니다.)

In [ ]:
os.environ.setdefault("OPENAI_API_KEY", "sk-dummy-not-used-in-this-notebook")

for name in sorted(os.listdir(SRC)):
    path = os.path.join(SRC, name)
    # import를 한 번이라도 하면 SRC 안에 __pycache__ 디렉터리가 생긴다.
    # 아래에서 open()으로 줄 수를 세므로, 파일이 아닌 것은 먼저 걸러낸다.
    if not os.path.isfile(path):
        continue
    print(f"  {name:<16} {len(open(path, encoding='utf-8').read().splitlines()):>3}줄")

| 파일 | 역할 |
|---|---|
| `reader.py` | PostgreSQL에서 원본을 꺼내온다 (입력) |
| `preprocess.py` | 정제 → 청킹 → 키워드 → 색인 (본체) |
| `config.py` | PostgreSQL과 OpenSearch 주소를 모두 들고 있다 |

`config.py`가 두 시스템 주소를 다 갖고 있다는 게 이 프로젝트의 성격을 말해줍니다.
**한쪽에서 읽어 다른 쪽으로 옮기는 다리** 역할입니다.

## 1. `reader.py` — 원본을 어떤 모양으로 받아오나

먼저 입력부터 봅니다.

In [ ]:
show("reader.py", grep="class RawDocument|    id|    url|    content_type|    text_content|    binary_content|SELECT|def fetch")

**코드에서 짚을 곳**

`RawDocument`는 **dataclass**입니다. 딕셔너리를 써도 되는데 왜 굳이 클래스를 만들었을까요?

```python
raw_doc["text_content"]   # 딕셔너리: 오타 나도 실행 전엔 모름
raw_doc.text_content      # dataclass: 자동완성 되고, 오타면 바로 에러
```

특히 `content_type`처럼 `'html'`이나 `'pdf'` 둘 중 하나만 오는 값은,
어떤 값이 가능한지 클래스 정의만 보면 알 수 있습니다. **데이터의 모양을 코드에 적어두는 것**이죠.

PostgreSQL 대신 SQLite로 같은 테이블을 만들고, 가짜 원본 3건을 넣어보겠습니다.

In [ ]:
import sqlite3

from reader import RawDocument

conn = sqlite3.connect(":memory:")
conn.execute("""
CREATE TABLE crawled_documents (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    url TEXT NOT NULL UNIQUE,
    content_type TEXT NOT NULL,
    text_content TEXT,
    binary_content BLOB
)
""")

conn.execute(
    "INSERT INTO crawled_documents (url, content_type, text_content) VALUES (?, ?, ?)",
    (
        "https://ex.com/rules/work-hours",
        "html",
        "제9조(근로시간)\n\n\n\n1주간의  소정근로시간은   휴게시간을 제외하고 40시간으로 한다....\n"
        "제10조(휴게)!!!! 회사는 근로시간이 4시간인 경우에는 30분 이상의 휴게시간을 부여한다.",
    ),
)
conn.execute(
    "INSERT INTO crawled_documents (url, content_type, text_content) VALUES (?, ?, ?)",
    ("https://ex.com/rules/empty", "html", "   "),  # 빈 문서 — 나중에 건너뛰어지는지 볼 것
)
conn.commit()

rows = conn.execute("SELECT id, url, content_type, text_content, binary_content FROM crawled_documents").fetchall()
raw_docs = [RawDocument(id=r[0], url=r[1], content_type=r[2], text_content=r[3], binary_content=r[4]) for r in rows]

for doc in raw_docs:
    print(f"[{doc.id}] {doc.url} ({doc.content_type}) {len(doc.text_content or '')}자")

## 2. `clean_text()` — 정규식 4줄이 하는 일

정제 함수를 읽고, 각 줄이 무엇을 잡는지 하나씩 확인해보겠습니다.

In [ ]:
show("preprocess.py", grep="re\\.sub")

In [ ]:
from preprocess import clean_text

dirty = raw_docs[0].text_content
print("=== 정제 전 ===")
print(repr(dirty))
print("\n=== 정제 후 ===")
print(repr(clean_text(dirty)))

**결과 읽는 법** — 연속 공백이 하나로, 빈 줄 4개가 2개로, `....`이 `.`으로, `!!!!`가 `!`로 줄었습니다.

**이게 왜 중요할까요?** 다음 단계인 청킹은 `\n\n`(빈 줄)을 우선 경계로 삼아 자릅니다.
빈 줄이 아무 데나 흩어져 있으면 **의미와 상관없는 곳에서 잘립니다.**
정제는 청킹의 품질을 위한 사전 작업입니다.

각 정규식을 직접 만져보면 감이 더 빨리 옵니다.

In [ ]:
import re

samples = {
    r"[ \t]+ -> ' '": ("공백    이     많은   문장", r"[ \t]+", " "),
    r"\n{3,} -> \n\n": ("줄\n\n\n\n\n바꿈", r"\n{3,}", "\n\n"),
    r"\.{2,} -> '.'": ("끝났습니다......", r"\.{2,}", "."),
}
for label, (text, pattern, repl) in samples.items():
    print(f"{label:<20} {repr(text):<28} -> {repr(re.sub(pattern, repl, text))}")

## 3. `extract_pdf_text()` — PDF를 파일로 저장하지 않고 읽기

PDF는 앞 프로젝트에서 **바이너리 그대로** 저장됐습니다. 여기서 처음으로 글자를 뽑아냅니다.

In [ ]:
show("preprocess.py", grep="def extract_pdf_text|BytesIO|fitz.open|get_text")

**코드에서 짚을 곳** — 눈여겨볼 곳은 `io.BytesIO`입니다.

DB에서 꺼낸 PDF는 **메모리 위의 바이트 덩어리**지, 디스크의 파일이 아닙니다.
보통 PDF 라이브러리는 파일 경로를 받는데, 그러려면 임시 파일로 저장했다가 읽고 지워야 합니다.
`BytesIO`는 바이트 덩어리를 "파일인 척" 감싸주기 때문에 그 왕복이 통째로 사라집니다.

실제로 PDF를 하나 만들어서 넣어보겠습니다.

In [ ]:
import fitz

from preprocess import extract_pdf_text

# 연습용 PDF를 메모리에 만듭니다.
doc = fitz.open()
page = doc.new_page()
page.insert_text((72, 100), "Article 11 (Remote Work)", fontsize=14)
page.insert_text((72, 130), "Employees may work remotely up to 2 days a week.", fontsize=11)
pdf_bytes = doc.tobytes()
doc.close()

print("PDF 크기:", len(pdf_bytes), "바이트")
print("추출된 텍스트:")
print(extract_pdf_text(pdf_bytes))

> ⚠️ **한글 PDF 주의** — 여기서 영어로 만든 이유는 기본 폰트에 한글이 없기 때문입니다.
> 실제 규정 PDF는 한글이 잘 뽑히지만, **스캔한 이미지 PDF는 글자가 하나도 안 나옵니다.**
> 그때는 OCR이 필요하고, 그건 다음 프로젝트([`03_document_input`](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/project-walkthrough/03_document_input/03_document_input.ipynb))의 주제입니다.
>
> 추출 결과가 비어 있으면 "라이브러리가 고장 났나?"가 아니라 **"이거 이미지 PDF구나"**를 먼저 의심하세요.

## 4. `extract_keywords()` — 왜 형태소 분석이 필요한가

한국어 검색에서 가장 자주 걸리는 함정입니다.

In [ ]:
text = "연차휴가를 신청하려면 휴가 신청서를 제출한다. 휴가는 3일 전까지 신청해야 한다."

print("공백으로 자르면:")
print("  ", text.split())

**결과 읽는 법**

`휴가를`, `휴가는`, `휴가`가 **전부 다른 단어로 잡힙니다.** 사람 눈엔 다 "휴가"인데요.
사용자가 "휴가"로 검색하면 `휴가를`이 들어간 문서를 놓칠 수 있습니다.

형태소 분석기(Kiwi)는 조사를 떼어내고 명사만 남깁니다.

In [ ]:
show("preprocess.py", grep="def extract_keywords|_kiwi|NNG|ranked|return \\[noun")

In [ ]:
from preprocess import extract_keywords

print("형태소 분석 결과 (빈도순 상위 키워드):")
print("  ", extract_keywords(text))

**결과 읽는 법**

`휴가`가 하나로 합쳐져서 빈도 3으로 잡혔습니다.

**`NNG`/`NNP`만 고른 이유**는 일반명사·고유명사가 문서의 주제를 가장 잘 대표하기 때문입니다.
동사나 조사는 어느 문서에나 나오니 구분에 도움이 안 됩니다.

> 💡 여기서 뽑은 키워드는 청크의 **메타데이터**로 붙습니다. 나중에
> "휴가 관련 청크만 필터링해서 검색" 같은 걸 할 수 있게 되죠.
> `_kiwi`를 모듈 레벨에서 한 번만 만든 것도 이유가 있습니다. Kiwi는 초기화할 때 사전을 읽느라
> 느려서, 청크마다 새로 만들면 전체 처리 시간이 몇 배가 됩니다.

## 5. `split_into_chunks()` — 전체를 이어붙여 돌리기

여기까지의 함수들이 하나로 모이는 지점입니다. 코드를 읽고 실제로 돌려봅시다.

In [ ]:
show("preprocess.py", grep="CHUNK_SIZE|CHUNK_OVERLAP|def split_into_chunks|splitter =|separators|건너뜀|metadata\\[")

In [ ]:
from preprocess import split_into_chunks

chunks = split_into_chunks(raw_docs)
print(f"\n총 {len(chunks)}개 청크\n")
for chunk in chunks:
    print("-", repr(chunk.page_content[:60]))
    print("  메타데이터:", chunk.metadata)

**결과 읽는 법** — 빈 문서(`/rules/empty`)는 **건너뜀**으로 처리되어 청크가 안 만들어졌습니다.
텍스트가 없는 문서를 그대로 색인하면 빈 벡터가 들어가서 검색 결과를 오염시킵니다.

## 6. `CHUNK_SIZE = 500`은 어디서 나온 숫자인가

`preprocess.py`는 500자, `rag-regulation-example/ingest.py`는 (예전엔) 1000자를 썼습니다.
**정답이 있는 값이 아니라 트레이드오프**입니다. 직접 비교해보죠.

In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

long_text = (clean_text(raw_docs[0].text_content) + "\n\n") * 12

for size in (200, 500, 1000):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=size, chunk_overlap=int(size * 0.15), separators=["\n\n", "\n", ". ", " ", ""]
    )
    pieces = splitter.split_documents([Document(page_content=long_text)])
    avg = sum(len(p.page_content) for p in pieces) / len(pieces)
    print(f"chunk_size={size:<5} 청크 {len(pieces):>3}개, 평균 {avg:>6.1f}자")

| 청크가 작으면 | 청크가 크면 |
|---|---|
| 검색이 정밀해짐 (딱 필요한 조각만) | 문맥이 살아있음 |
| 문맥이 잘려서 의미를 잃을 수 있음 | 관련 없는 내용이 섞여 검색이 뭉개짐 |
| 조각 수가 늘어 임베딩 비용 증가 | AI에게 넘기는 토큰이 늘어 비용 증가 |

`preprocess.py`가 500을 고른 이유는 코드 주석에 적혀 있습니다 —
**조항 번호가 촘촘한 규정 문서**라서 작게 잘라야 "질문과 관련된 조항 하나"를 집어낼 수 있다는 것.

> 💡 그런데 이 접근에는 더 나은 대안이 있습니다. 규정 문서에는 **사람이 이미 정해둔 경계(조항)**가
> 있는데, 자로 재서 500자씩 자르면 그 경계를 무시하게 됩니다.
> 조 단위로 자르는 방법은 [`04_rag_regulation`](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/project-walkthrough/04_rag_regulation/04_rag_regulation.ipynb)에서 다룹니다.
> **거기서 "왜 고정 길이 청킹을 그만뒀는지"가 이 노트북의 답입니다.**

## 7. `index_chunks()` — OpenSearch 없이 감 잡기

마지막 단계는 임베딩 후 색인입니다. 이 부분만은 외부 서비스가 필요해서 그대로 돌릴 수 없습니다.

In [ ]:
show("preprocess.py", grep="def index_chunks|OpenAIEmbeddings|from_documents|engine=|space_type|Indexed")

**코드에서 짚을 곳**

`OpenSearchVectorSearch.from_documents()` 한 줄이 사실 세 가지를 합니다.

1. 청크를 하나씩 OpenAI에 보내 **벡터로 변환**
2. `원문 + 벡터 + 메타데이터`를 묶어서
3. OpenSearch 인덱스에 저장

벡터가 뭘 하는 건지는 API 없이도 흉내 낼 수 있습니다. TF-IDF로 대신해봅시다.
(원리는 같습니다 — 글을 숫자 벡터로 바꾸고, 가까운 것을 찾는다.)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

corpus = [c.page_content for c in chunks]
vectorizer = TfidfVectorizer()
matrix = vectorizer.fit_transform(corpus)
print("청크", matrix.shape[0], "개 -> 각각", matrix.shape[1], "차원 벡터")

question = "휴게시간은 얼마나 주나요?"
scores = cosine_similarity(vectorizer.transform([question]), matrix)[0]

for idx in scores.argsort()[::-1]:
    print(f"\n유사도 {scores[idx]:.3f}")
    print("  ", corpus[idx][:70])

> ⚠️ **TF-IDF는 단어가 겹쳐야만 점수가 나옵니다.** "휴게시간"이라고 물었으니 걸렸지,
> "쉬는 시간"이라고 물었으면 0점이 나왔을 겁니다.
> OpenAI 임베딩은 **뜻이 비슷하면** 글자가 안 겹쳐도 찾아냅니다. 그 차이가 임베딩에 돈을 쓰는 이유입니다.
>
> 직접 확인해보세요. 위 질문을 "쉬는 시간"으로 바꾸면 어떻게 되나요?

## 정리

이 프로젝트는 **원본과 검색엔진 사이의 다리**입니다.

```
RawDocument (PDF/HTML 섞임)
   -> raw_document_to_text()   형식 통일 + 정제
   -> split_into_chunks()      500자 청킹 + 키워드 태깅
   -> index_chunks()           임베딩 -> OpenSearch
```

건진 것들:

| 결정 | 이유 |
|---|---|
| PDF/HTML을 텍스트로 통일 | 뒤 단계가 형식을 신경 쓰지 않게 |
| `BytesIO`로 메모리에서 처리 | 임시 파일 왕복 제거 |
| 정제를 청킹보다 **먼저** | 잡음이 청크 경계를 망치지 않게 |
| 형태소 분석으로 키워드 | 한국어 조사 때문에 공백 분리는 안 통함 |
| `_kiwi`를 모듈 레벨에 한 번만 | 초기화 비용이 큼 |
| 빈 문서 건너뛰기 | 빈 벡터가 검색을 오염시킴 |

**가장 기억할 것**: **검색 품질은 색인 전에 결정됩니다.**
잡음이 낀 채로 자르면 청크 경계가 엉뚱한 곳에서 갈리고, 그건 나중에 어떤 검색 기법으로도 복구되지 않습니다.

**스스로 확인해보기**

- [ ] 정제를 청킹보다 먼저 하는 이유를 설명할 수 있다
- [ ] 한국어에서 공백으로 단어를 자르면 안 되는 이유를 예를 들어 말할 수 있다
- [ ] `chunk_size`를 키우면/줄이면 각각 무엇이 좋아지고 나빠지는지 안다
- [ ] PDF에서 텍스트가 하나도 안 나올 때 무엇을 먼저 의심해야 하는지 안다
- [ ] TF-IDF 검색이 "쉬는 시간"으로 물었을 때 왜 실패하는지 직접 확인했다

## 연습 문제

**1. 청크에 순번 붙이기**
지금 메타데이터에는 `source`와 `keywords`만 있습니다. "이 문서의 3번째 청크"라는 정보를 넣으려면?
그게 있으면 무엇을 할 수 있을까요? (힌트: 앞뒤 청크를 같이 보여주기)

**2. 정제 규칙 추가하기**
크롤링한 HTML에는 `[목차로]`, `인쇄하기` 같은 UI 텍스트가 섞여 들어옵니다.
`clean_text()`에 이런 걸 걸러내는 규칙을 추가해보세요. **과하게 지우면 생기는 문제**도 생각해보세요.

**3. 키워드를 검색에 실제로 써보기**
`extract_keywords()`가 뽑은 키워드는 지금 저장만 되고 검색에 쓰이지 않습니다.
질문에서도 명사를 뽑아 청크 키워드와 겹치는 개수로 점수를 매기는 검색을 만들어보고,
위 TF-IDF 결과와 비교해보세요.

**해설/정답**: [02_preprocess_solutions.ipynb](02_preprocess_solutions.ipynb)

## 다음 단계

- [`03_document_input`](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/project-walkthrough/03_document_input/03_document_input.ipynb) — 사용자가 올린 서류 사진을 처리하는 별도 입력 경로
- [`04_rag_regulation`](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/project-walkthrough/04_rag_regulation/04_rag_regulation.ipynb) — 색인된 문서로 실제 질의응답. **고정 길이 청킹의 대안**도 여기서 다룹니다
- 라이브러리 자체 실습: [`rag-pipeline-practice/02_text_chunking`](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/rag-pipeline-practice/02_text_chunking/02_text_chunking.ipynb)